In [7]:
#Cau 1

import numpy as np
from scipy.spatial.distance import cdist

# Khởi tạo tập dữ liệu mẫu 3 cụm như tài liệu hướng dẫn
means = [[2, 2], [9, 2], [4, 9]]
cov = [[2, 0], [0, 2]]
n_samples = 150
n_cluster = 3

X0 = np.random.multivariate_normal(means[0], cov, n_samples)
X1 = np.random.multivariate_normal(means[1], cov, n_samples)
X2 = np.random.multivariate_normal(means[2], cov, n_samples)
X = np.concatenate((X0, X1, X2), axis=0)

def kmeans_init_centers(X, n_cluster):
    # Chọn ngẫu nhiên k mẫu dữ liệu làm tâm cụm ban đầu không trùng lặp
    return X[np.random.choice(X.shape[0], n_cluster, replace=False)]

def kmeans_predict_labels(X, centers):
    # Tính khoảng cách từ các điểm tới các tâm và gán vào tâm gần nhất
    D = cdist(X, centers)
    return np.argmin(D, axis=1)

def kmeans_update_centers(X, labels, n_cluster):
    # Cập nhật tâm cụm mới bằng cách tính trung bình cộng tọa độ các điểm thuộc cụm
    centers = np.zeros((n_cluster, X.shape[1]))
    for k in range(n_cluster):
        Xk = X[labels == k, :]
        centers[k, :] = np.mean(Xk, axis=0)
    return centers

def kmeans_has_converged(centers, new_centers):
    # Kiểm tra xem vị trí tâm cụm đã hội tụ (không thay đổi) hay chưa
    return (set([tuple(a) for a in centers]) == set([tuple(a) for a in new_centers]))

def run_kmeans(X, n_cluster):
    centers = kmeans_init_centers(X, n_cluster)
    labels = np.zeros(X.shape[0])

    while True:
        labels = kmeans_predict_labels(X, centers)
        new_centers = kmeans_update_centers(X, labels, n_cluster)
        if kmeans_has_converged(centers, new_centers):
            break
        centers = new_centers
    return centers, labels

if __name__ == "__main__":
    final_centers, final_labels = run_kmeans(X, n_cluster)
    print("--- THUẬT TOÁN K-MEANS HOÀN THÀNH ---")
    print("Tọa độ các tâm cụm hội tụ cuối cùng:\n", final_centers)

--- THUẬT TOÁN K-MEANS HOÀN THÀNH ---
Tọa độ các tâm cụm hội tụ cuối cùng:
 [[4.25980667 9.03758568]
 [9.00248929 2.18219616]
 [1.83039415 1.96826625]]


In [9]:
#Cau 2
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, train_test_split

# Khởi tạo tập dữ liệu phân loại gồm 200 mẫu chia thành 4 lớp riêng biệt
X, y = make_blobs(n_samples=200, n_features=2, centers=4, cluster_std=1.2, random_state=4)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

# CÁCH 1: Hàm thuật toán KNN tự xây dựng từ đầu (Học máy thủ công)
def KNN_scratch(X_train, X_test, y_train, k):
    num_test = X_test.shape[0]
    num_train = X_train.shape[0]
    y_pred = np.zeros((num_test, num_train))

    for i in range(num_test):
        for j in range(num_train):
            # Tính khoảng cách Euclidean giữa điểm kiểm tra và điểm huấn luyện
            y_pred[i, j] = np.sqrt(np.sum(np.power(X_test[i, :] - X_train[j, :], 2)))

    results = []
    for i in range(len(y_pred)):
        zipped = zip(y_pred[i, :], y_train)
        res = sorted(zipped, key=lambda x: x[0])
        results_topk = res[:k]

        classes = {}
        for _, cls_label in results_topk:
            cls_label = int(cls_label)
            classes[cls_label] = classes.get(cls_label, 0) + 1

        results.append(max(classes, key=classes.get))
    return np.array(results)

if __name__ == "__main__":
    print("--- ĐÁNH GIÁ TRỌNG SỐ K TRONG KNN ---")

    # Thực thi cách 1 với giá trị k cố định lựa chọn theo trực quan cảm tính
    k_manual = 5
    predictions_scratch = KNN_scratch(X_train, X_test, y_train, k_manual)
    accuracy_scratch = np.mean(predictions_scratch == y_test)
    print(f"Cách 1: Độ chính xác với k chọn thủ công ({k_manual}) = {accuracy_scratch * 100:.2f}%")

    # CÁCH 2: Tìm kiếm giá trị K tối ưu tự động bằng công cụ Grid Search chéo hóa dữ liệu (CV)
    param_grid = {'n_neighbors': np.arange(1, 15)}
    grid_search = GridSearchCV(estimator=KNeighborsClassifier(), param_grid=param_grid, cv=5)
    grid_search.fit(X, y)

    print(f"Cách 2: Tham số k tốt nhất tìm thấy tự động = {grid_search.best_params_['n_neighbors']}")
    print(f"Độ chính xác tốt nhất trên tập dữ liệu đạt: {grid_search.best_score_ * 100:.2f}%")

--- ĐÁNH GIÁ TRỌNG SỐ K TRONG KNN ---
Cách 1: Độ chính xác với k chọn thủ công (5) = 96.00%
Cách 2: Tham số k tốt nhất tìm thấy tự động = 5
Độ chính xác tốt nhất trên tập dữ liệu đạt: 97.00%


In [2]:
#Cau 3
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.neighbors import KNeighborsClassifier

def demo_application():
    print("================================================")
    print(" HỆ THỐNG DEMO CHƯƠNG TRÌNH AI - TUẦN 5")
    print("================================================")
    print("1. Chạy Demo mô hình Gom cụm dữ liệu K-Means")
    print("2. Chạy Demo mô hình Phân loại dữ liệu K-NN")
    print("3. Thoát hệ thống")
    choice = input("Nhập tùy chọn ứng dụng của bạn (1-3): ")

    if choice == '1':
        print("\n[ĐANG CHẠY K-MEANS] Tự động gom cụm tập dữ liệu tọa độ địa lý...")
        X_data, _ = make_blobs(n_samples=100, centers=3, random_state=42)
        # Khởi tạo nhanh cấu hình thuật toán K-Means
        from sklearn.cluster import KMeans
        model_kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
        model_kmeans.fit(X_data)
        print("Tâm các khu vực mật độ cao định vị được:\n", model_kmeans.cluster_centers_)
        print("Gom cụm dữ liệu hoàn tất thành công!")

    elif choice == '2':
        print("\n[ĐANG CHẠY K-NN] Dự đoán nhãn phân loại khách hàng tiềm năng...")
        X_train, y_train = make_blobs(n_samples=80, centers=2, random_state=10)
        knn = KNeighborsClassifier(n_neighbors=3)
        knn.fit(X_train, y_train)

        # Điểm dữ liệu cần dự đoán nhãn
        new_point = np.array([[0.5, 2.3]])
        predicted_class = knn.predict(new_point)
        print(f"Tọa độ khách hàng mới nhập vào hệ thống: {new_point[0]}")
        print(f"Kết quả phân loại nhóm khách hàng: Lớp {predicted_class[0]}")

    else:
        print("Thoát chương trình ứng dụng demo.")

if __name__ == "__main__":
    demo_application()

 HỆ THỐNG DEMO CHƯƠNG TRÌNH AI - TUẦN 5
1. Chạy Demo mô hình Gom cụm dữ liệu K-Means
2. Chạy Demo mô hình Phân loại dữ liệu K-NN
3. Thoát hệ thống
Nhập tùy chọn ứng dụng của bạn (1-3): 2

[ĐANG CHẠY K-NN] Dự đoán nhãn phân loại khách hàng tiềm năng...
Tọa độ khách hàng mới nhập vào hệ thống: [0.5 2.3]
Kết quả phân loại nhóm khách hàng: Lớp 1


In [1]:
#Bai Tap Ve Nha
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier

# --- GIẢ LẬP DỮ LIỆU THỰC TẾ ---
# Tạo ngẫu nhiên dữ liệu 150 khách hàng cho mô hình K-Means
np.random.seed(42)
X_customers = np.random.rand(150, 2) * 100 # Thu nhập và điểm chi tiêu chạy từ 0-100

# Tạo ngẫu nhiên dữ liệu 100 hồ sơ vay tín dụng cho mô hình K-NN
X_credit = np.random.rand(100, 2) * 10
y_credit = np.array([1 if (x[0]*0.7 + x[1]*0.3) > 5 else 0 for x in X_credit])

def run_homework_4():
    print("--- [ĐANG CHẠY DEMO BÀI 4] ỨNG DỤNG DOANH NGHIỆP ---")

    # 1. Thực thi K-Means phân tách khách hàng thương mại điện tử
    print("\n[Tiến trình 1] Hệ thống đang phân khúc nhóm khách hàng bằng K-Means...")
    kmeans_app = KMeans(n_clusters=4, random_state=42, n_init='auto')
    customer_labels = kmeans_app.fit(X_customers)
    print(f"Đã phân loại thành công 150 khách hàng vào {kmeans_app.n_clusters} nhóm thị trường mục tiêu khác nhau.")
    print("Tâm điểm đặc trưng của các nhóm khách hàng:\n", kmeans_app.cluster_centers_)

    # 2. Thực thi K-NN tự động chấm điểm duyệt hồ sơ vay vốn ngân hàng
    print("\n[Tiến trình 2] Đang huấn luyện hệ thống xét duyệt tín dụng tự động K-NN...")
    knn_credit = KNeighborsClassifier(n_neighbors=5)
    knn_credit.fit(X_credit, y_credit)

    # Giả lập hồ sơ khách hàng mới nộp đơn
    new_applicant = np.array([[7.5, 4.2]]) # Điểm tín dụng cao, tỷ lệ nợ trung bình thấp
    approval_status = knn_credit.predict(new_applicant)

    status_text = "ĐƯỢC PHÊ DUYỆT VAY CO SỞ" if approval_status[0] == 1 else "BỊ TỪ CHỐI CHO VAY TÍN DỤNG"
    print(f"Hồ sơ xét duyệt mới có chỉ số: Điểm tín dụng={new_applicant[0][0]}, Tỷ lệ nợ={new_applicant[0][1]}")
    print(f"==> KẾT QUẢ ĐÁNH GIÁ CỦA MÔ HÌNH K-NN: {status_text}")

if __name__ == "__main__":
    run_homework_4()

--- [ĐANG CHẠY DEMO BÀI 4] ỨNG DỤNG DOANH NGHIỆP ---

[Tiến trình 1] Hệ thống đang phân khúc nhóm khách hàng bằng K-Means...
Đã phân loại thành công 150 khách hàng vào 4 nhóm thị trường mục tiêu khác nhau.
Tâm điểm đặc trưng của các nhóm khách hàng:
 [[78.35838438 74.54072743]
 [19.49237004 71.4889314 ]
 [24.38194753 20.66611157]
 [72.32618931 25.32169489]]

[Tiến trình 2] Đang huấn luyện hệ thống xét duyệt tín dụng tự động K-NN...
Hồ sơ xét duyệt mới có chỉ số: Điểm tín dụng=7.5, Tỷ lệ nợ=4.2
==> KẾT QUẢ ĐÁNH GIÁ CỦA MÔ HÌNH K-NN: ĐƯỢC PHÊ DUYỆT VAY CO SỞ
